In [3]:
!pip install ultralytics
!pip install cvzone

  Preparing metadata (setup.py) ... done
  Created wheel for cvzone: filename=cvzone-1.6.1-py3-none-any.whl size=26297 sha256=eccea70c18bbabfb1a558d1dec2cfc263230a6141381d3413b095ad9c0875fc3
  Stored in directory: /root/.cache/pip/wheels/2c/9f/b3/92e945ac4a71bf727a92463f38155cc5a4fa49c5010b38ec4c
Successfully built cvzone


In [2]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
import numpy as np
from ultralytics import YOLO
import cv2
import cvzone
import math
# from sort_tracker import Sort


cap = cv2.VideoCapture('/kaggle/input/iiuc-idcard-data/test/VID20241006095229.mp4')


if not cap.isOpened():
    print("Error: Could not open video.")
    exit()
    

model = YOLO("/kaggle/working/runs/detect/train13/weights/best.pt")

classNames = ['not_wearing_idcard','wearing_idcard']

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('iiuc-idcard.mp4', fourcc, fps, (width,height))

# tracker = Sort(max_age=20,min_hits=3,iou_threshold=0.3)

while True:
    success, img = cap.read()
    if not success:
        print("Error: Could not read frame.")
        break
#     img = cv2.resize(img,(1280,720))

    
    results = model(img, stream=True)
    for r in results:
        #bounding boxex
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            w, h = x2 - x1, y2 - y1 
            

            

            conf = math.ceil(box.conf[0] * 100) / 100
            class_name = int(box.cls[0])
            # print(class_name)
            if classNames[class_name] =='not_wearing_idcard':
                cvzone.cornerRect(img, (x1, y1, w, h),l=5,rt=2,colorR=(0,0,255),colorC=(0,0,150))
                cvzone.putTextRect(img, f"{classNames[class_name]}{conf}", (max(0, x1), max(35, y1)),
                                 scale=1,thickness=1,offset=4,colorR=(0,0,255))
            else:
                cvzone.cornerRect(img, (x1, y1, w, h),l=5,rt=2,colorC=(0,255,0),colorR=(0,150,0))
                cvzone.putTextRect(img, f"{classNames[class_name]}{conf}", (max(0, x1), max(35, y1)),
                                 scale=1,thickness=1,offset=4,colorR=(0,255,0),)
    #             currentArray = np.array([x1,y1,x2,y2,conf])
    #             detections = np.vstack((detections,currentArray))

    # resultsTracker = tracker.update(detections)

#     cv2.imshow("webcam", img)
#     cv2.waitKey(0)
    out.write(img)
cap.release()
out.release()


0: 384x640 6 not_wearing_idcards, 4 wearing_idcards, 53.5ms
Speed: 3.5ms preprocess, 53.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 not_wearing_idcards, 6 wearing_idcards, 10.0ms
Speed: 2.2ms preprocess, 10.0ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 not_wearing_idcards, 5 wearing_idcards, 13.7ms
Speed: 3.0ms preprocess, 13.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 not_wearing_idcards, 5 wearing_idcards, 10.1ms
Speed: 2.4ms preprocess, 10.1ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 not_wearing_idcards, 4 wearing_idcards, 12.6ms
Speed: 3.2ms preprocess, 12.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 not_wearing_idcards, 4 wearing_idcards, 10.2ms
Speed: 2.2ms preprocess, 10.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 not_wearing_idcards, 3 wearing_idcards

In [8]:
from ultralytics import YOLO

# Load a model
model = YOLO("/kaggle/working/runs/detect/train13/weights/last.pt")  # load a pretrained model (recommended for training)

# Train the model with 2 GPUs
results = model.train(data="/kaggle/input/iiuc-idcard-data/data.yaml", epochs=100, imgsz=640,lr)

Ultralytics 8.3.13 🚀 Python-3.10.14 torch-2.4.0 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=/kaggle/working/runs/detect/train13/weights/last.pt, data=/kaggle/input/iiuc-idcard-data/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train14, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False

train: Scanning /kaggle/input/iiuc-idcard-data/train/labels... 46 images, 0 backgrounds, 0 corrupt: 100%|██████████| 46/46 [00:00<00:00, 487.93it/s]

train: WARNING ⚠️ Cache directory /kaggle/input/iiuc-idcard-data/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))



/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
val: Scanning /kaggle/input/iiuc-idcard-data/valid/labels... 12 images, 0 backgrounds, 0 corrupt: 100%|██████████| 12/12 [00:00<00:00, 360.59it/s]

val: WARNING ⚠️ Cache directory /kaggle/input/iiuc-idcard-data/valid is not writeable, cache not saved.


...

                   all         12        118      0.741      0.728      0.761      0.326



50 epochs completed in 0.044 hours.
Optimizer stripped from runs/detect/train14/weights/last.pt, 19.2MB
Optimizer stripped from runs/detect/train14/weights/best.pt, 19.2MB

Validating runs/detect/train14/weights/best.pt...
Ultralytics 8.3.13 🚀 Python-3.10.14 torch-2.4.0 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLO11s summary (fused): 238 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  7.42it/s]


                   all         12        118      0.762      0.727      0.808      0.343
    not_wearing_idcard         11         50      0.746       0.68      0.753      0.275
        wearing_idcard         12         68      0.778      0.773      0.862       0.41
Speed: 0.2ms preprocess, 5.1ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs/detect/train14


lr/pg0,▁▂▂▃▃▄▄▅▅▅▆▆▇▇▇▇███████████▇▇▆▆▅▅▄▄▃▃▂▂▁
lr/pg1,▁▂▂▃▃▄▄▅▅▅▆▆▇▇▇▇██████████▇▇▇▆▆▅▅▅▄▃▃▂▂▁
lr/pg2,▁▂▂▃▃▄▄▅▅▅▆▆▇▇▇███████████▇▇▆▆▅▅▅▄▄▃▃▂▂▁
metrics/mAP50(B),▄▅▁▂▆▃▅▆▆▃▆▆▄▄▂▄▄▄▃▁▄▃▃▃▄▆▅▆██▆▇▇▃▁▂▂▃▃█
metrics/mAP50-95(B),▄▅▁▂▆▄▄▆▄▄▇▅▆▅▄▇▇▆▆▄▆▄▃▃▅▆▇███▇▇▇▆▆▅▄▅▅█
metrics/precision(B),▃▆▂▆▆▂▅█▅▆▆▅▆▆▆▅▃▄▂▁▆▄▃▃▃▆▆▆▄▄▃▃▃▂▃▅▁▂▂▄
metrics/recall(B),█▆▇▁▆▇▅▄▇▅█▆▅▆▄▆▆▅█▅▄▂▃▃▅▆▆▅▄▆▇▇█▇▅▄██▆▆
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
train/box_loss,▃▄▅▇▆▅▅▆▅▆▅█▇██▇▇█▆▄▇▇▄▅▆▅▆▄▂▃▂▄▄▁▄▄▃▂▂▁
